# `klygo.datasets.repartition`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Repartition.ipynb)

Flatten existing split structures safely and create a new deterministic partition without creating nested split hierarchies.

## Jump to an API

- [`datasets.repartition`](#datasetsrepartition)


In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release


## Setup workspace and sample data


In [ ]:
from tempfile import TemporaryDirectory
from PIL import Image
from klygo import datasets, files, media

# Create isolated tutorial workspace
tutorial_workspace = TemporaryDirectory(prefix="klygo-datasets-tut-")
root = files.path(tutorial_workspace.name)

# Create a sample flat YOLO dataset
sample_dataset = files.join(root, "sample_dataset")
images_dir = files.join(sample_dataset, "images")
labels_dir = files.join(sample_dataset, "labels")
files.mkdir(images_dir)
files.mkdir(labels_dir)

for i in range(4):
    cls = i % 2
    media.save(files.join(images_dir, f"img_{i}.jpg"), Image.new("RGB", (64, 48), color=(40 * i, 80, 120)), verbose=False)
    files.save(files.join(labels_dir, f"img_{i}.txt"), [f"{cls} 0.5 0.5 0.3 0.3"], verbose=False)

files.save(
    files.join(sample_dataset, "data.yaml"),
    ["names:", "  0: cat", "  1: dog", "nc: 2"],
    verbose=False,
)
print("Sample dataset ready:", sample_dataset)


### datasets.repartition

Demo and practical usage for `datasets.repartition`.


In [ ]:
# First create an initial partition
p1 = datasets.partition(sample_dataset, files.join(root, "p1"), (0.5, 0.25, 0.25), verbose=False)

# Repartition with new ratios (70% train, 30% val)
repartitioned = datasets.repartition(
    source=p1,
    target=files.join(root, "repartitioned.zip"),
    ratios=(0.7, 0.3),
    format="yolo",
    overwrite=True,
    verbose=False,
)
print("Repartitioned archive:", repartitioned)
print("New split distribution:", datasets.info(repartitioned)["splits"])


## Put it together


In [ ]:
# Complete end-to-end workflow demonstrating datasets.repartition
print("Executing complete workflow for datasets.repartition...")
stats = datasets.info(sample_dataset, format="yolo", verbose=False)
print("Dataset classes:", stats["classes"])
print("Dataset images:", stats["total_images"])


## Remember

`datasets.repartition()` flattens previous split prefixes safely so repeated partitioning operations do not corrupt filenames or nest folder structures.

See [Datasets overview](Document_Datasets_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Overview.ipynb)) for full module documentation.


In [ ]:
tutorial_workspace.cleanup()
